# gradient-routing-slm: one run on a Kaggle T4
Settings: **Accelerator = GPU T4** (x1 or x2, only one is used), **Internet = On**. Edit the first cell, then Run All.
The last cell writes a zip to `/kaggle/working/`. Unzip it into `results/<name>/` in the repo, using the zip's name.

In [ ]:
ROUTING_MODE = "dense"   # dense (baseline) or gated (v3); the older batch, token, label and assembly modes still work
GROUPS = "fixed"         # gated mode: fixed or evolved
ROUTE_SMOOTH = 0.9       # gated mode: causal EMA of group scores along the sequence (0 = none)
SEED = 0                 # model init and batch order; the dataset and val sets stay fixed
MAX_ITERS = 5000
EXTRA_ENV = {}           # any other run_all.sh setting, e.g. {"RESOLUTION": "0.5"} for assembly mode
REPO_URL = "https://github.com/sandwichhlord/gradient-routing-slm.git"

In [ ]:
!pip install -q -U tiktoken datasets igraph leidenalg

In [ ]:
!rm -rf /tmp/repo && git clone --depth 1 {REPO_URL} /tmp/repo
%cd /tmp/repo
!git log --oneline -1
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import json, os, shutil, subprocess
import torch
assert torch.cuda.is_available(), "No GPU: set Accelerator = GPU T4 in the notebook settings"


def stream(cmd, log, env=None):
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
    for line in p.stdout:
        print(line, end="")
        log.write(line)
    return p.wait()


smooth = f"{ROUTE_SMOOTH:g}"
name = f"gated_{GROUPS}" if ROUTING_MODE == "gated" else ROUTING_MODE
tag = (f"_sm{smooth}" if smooth != "0.9" else "") + "".join(f"_{k.lower()}{v}" for k, v in EXTRA_ENV.items())
RUN = f"{name}{tag}_s{SEED}"     # e.g. gated_evolved_sm0_s0
OUT = f"out_{RUN}"
os.makedirs(OUT, exist_ok=True)
env = {**os.environ, "ROUTING_MODE": ROUTING_MODE, "GATE_GROUPS": GROUPS, "ROUTE_SMOOTH": smooth, "SEED": str(SEED),
       "MAX_ITERS": str(MAX_ITERS), "OUT_DIR": OUT, **{k: str(v) for k, v in EXTRA_ENV.items()}}
with open("run.log", "w") as log:
    rc = stream(["bash", "run_all.sh"], log, env)
shutil.copy("run.log", f"{OUT}/run.log")
if os.path.exists("data/out/meta.json"):
    shutil.copy("data/out/meta.json", f"{OUT}/data_meta.json")
    mode = json.load(open("data/out/meta.json"))["mode"]
    print()
    print("DATA MODE:", mode, "" if mode == "real" else " <-- WARNING: HF download failed, results are on synthetic data!")
print("exit code:", rc, "(the last cell still zips whatever was produced)")

In [ ]:
from IPython.display import Image, display
for name in ["loss_curves.png", "ablation_heatmap.png", "routing_by_domain.png", "assembly_health.png", "assembly_heatmap.png"]:
    if os.path.exists(f"{OUT}/{name}"):
        print(name)
        display(Image(f"{OUT}/{name}"))
for name in ["summary.txt", "ablation_summary.txt", "context_sensitivity.txt", "damage_summary.txt",
             "failure_trace_summary.txt", "unlearn_summary.txt", "assembly_summary.txt", "samples.txt"]:
    if os.path.exists(f"{OUT}/{name}"):
        print("---", name)
        print(open(f"{OUT}/{name}").read())

In [ ]:
zip_path = shutil.make_archive(f"/kaggle/working/{RUN}", "zip", OUT)
print("wrote", zip_path, f"({os.path.getsize(zip_path) / 1e6:.0f} MB)")
assert rc == 0, f"the run failed with exit code {rc}: see run.log inside the zip"